# Clasificación por Arboles de Decisión (ejemplo 2)

## Los datos

Se utilizará otro conjunto de datos bastante conocido, el de "Palmer Penguins", ya que es lo suficientemente simple para permitir conocer cómo el modificar los hiperparámetros pueden cambiar los resultados de clasificación.


<img src="penguin.jpg" style="max-width:400px">

Estos datos fueron recopilados, y hechos disponibles, por la Dra. Kristen Gorman y la estación Palmer, Antártica LTER, un miembro de la Long Term Ecological Research Network (LTER).

Gorman KB, Williams TD, Fraser WR (2014) Ecological Sexual Dimorphism and Environmental Variability within a Community of Antarctic Penguins (Genus Pygoscelis). PLoS ONE 9(3): e90081. doi:10.1371/journal.pone.0090081

Resumen:
Hay dos archivos CSV.  Para cursos introductorios, probablemente es mejor el (penguins_size.csv).

* penguins_size.csv: Datos simplificados del los conjuntos de datos originales de pinguinos.  Contiene las variables:

    * species: especie de pinguino (Chinstrap, Adélie, or Gentoo)
    * culmen_length_mm: longitud del pico (mm)
    * culmen_depth_mm: profundidad del pico (mm)
    * flipper_length_mm: longitud de aleta (mm)
    * body_mass_g: masa del cuerpo (g)
    * island: nombre de la isla (Dream, Torgersen, or Biscoe) en el archipiélago Palmer (Antárctica)
    * sex: sexo del pinguino  

Nota: El culmen es la orilla superior del pico del un pájaro

**La meta es crear un modelo que pueda ayudar a predecir la especie de un pinguino basado en los atributos físicos, luego se puede utilizar ese modelo para ayudar a los investigadores a clasificar los pinguinos en el campo, en vez de tener que contar con un biólogo experimentado**

## Importaciones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("penguins_size.csv")

In [ ]:
df.head()

## Análisis exploratorio de datos (EDA)

### Datos Faltantes

Recordar que el propósito es crear un modelo para uso futuro, así que aquellas observaciones a las que les falta información crucial no serán de ayuda para esta tarea.  Esto es cierto especialmente porque para observaciones futuras se asume que las investigaciones podrán captar toda la información relevante.

In [ ]:
df.info()

In [ ]:
df.isna().sum()

¿Si eliminamos estos casos, qué porcentaje del total representa?

In [ ]:
100*(10/344)

In [ ]:
df = df.dropna()

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df['sex'].unique()

In [ ]:
df['island'].unique()

In [ ]:
df[df['sex'] == '.']

In [ ]:
df.query("sex == '.'")

In [ ]:
df.tail()

In [ ]:
df.reset_index(inplace = True)

In [ ]:
df[df['species'] == 'Gentoo'].groupby('sex').describe()

In [ ]:
df[df['species'] == 'Gentoo'].groupby('sex').describe().transpose()

Comparando los diferentes atributos, podríamos pensar que este caso es de sexo femenino.  El cambio es más fácil usando el método "at" de Pandas

In [ ]:
df.at[327, 'sex'] = 'FEMALE'

In [ ]:
df.loc[327]

Si no tuviéramos la seguridad necesaria, podríamos simplemente ejecutar la siguiente instrucción 

In [ ]:
#df = df[df['sex'] != '.']

## Visualización

In [ ]:
sns.scatterplot(x = 'culmen_length_mm', 
                y = 'culmen_depth_mm',
                data = df,
                hue = 'species',
                palette = 'Dark2')

In [ ]:
sns.pairplot(df, hue = 'species', palette = 'Dark2')

In [ ]:
sns.catplot(x = 'species', 
            y = 'culmen_length_mm',
            data = df, 
            kind = 'box',
            col = 'sex',
            palette = 'Dark2')

## Ingeniería de Atributos (Feature Engineering)

Aunque la gente de Sklearn está trabajando para manejar variables categóricas, tal cuál, aún no está disponible.  Por lo tanto es necesario codificarlas.  Las siguientes dos líneas son solo para probar, no se cambia nada

In [ ]:
pd.get_dummies(df)

In [ ]:
pd.get_dummies(df.drop('species', axis = 1), drop_first = True)

In [ ]:
df.drop("index", axis = 1, inplace = True)

## División Entrenamiento | Prueba

In [ ]:
X = pd.get_dummies(df.drop('species', axis = 1), drop_first = True)
y = df['species']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_entreno, X_prueba, y_entreno, y_prueba = train_test_split(X, y, test_size=0.3, random_state=101)

**NOTA**  Como en este algoritmo solo se trata de ver si una observación pertenece a una clase, o no, no es necesario normalizar las variables

# Clasificador por Arbol de Decisión

## Hiperparámeteros por default

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
modelo = DecisionTreeClassifier()

In [ ]:
modelo.fit(X_entreno, y_entreno)

In [ ]:
y_predicciones = modelo.predict(X_prueba)

## Evaluación

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
matriz_confusion = confusion_matrix(y_prueba, y_predicciones)
matriz_confusion

In [ ]:
despliegue = ConfusionMatrixDisplay(confusion_matrix = matriz_confusion)
despliegue.plot()

In [ ]:
print(classification_report(y_prueba, y_predicciones))

In [ ]:
modelo.feature_importances_

In [ ]:
pd.DataFrame(index = X.columns, data = modelo.feature_importances_, 
             columns=['Importancia_Atributos']).sort_values('Importancia_Atributos')

In [ ]:
sns.boxplot(x = 'species', y = 'body_mass_g', data = df)

## Visualizar el Arbol

Esta función es bastante reciente, puede que quiera revisar los documentos en-línea:

Documentación en-línea: https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html

In [ ]:
from sklearn.tree import plot_tree

In [ ]:
plt.figure(figsize = (12, 8))
plot_tree(modelo);

In [ ]:
plt.figure(figsize = (12, 8), dpi = 150)
plot_tree(modelo, filled = True, feature_names = list(X.columns));

## Reportando los Resultados del Modelo

Para empezar a experimentar con hiperparámetros, se crea una función que reporta el resultado de la clasificación y grafica el árbol.

In [ ]:
def reporte_modelo(modelo):
    modelo_predicciones = modelo.predict(X_prueba)
    print(classification_report(y_prueba,modelo_predicciones))
    print('\n')
    plt.figure(figsize = (12, 8), dpi = 150)
    plot_tree(modelo, filled = True, feature_names = list(X.columns));

## Entendiendo los Hiperparámeteros

### Profundidad Máxima (Max Depth)

In [ ]:
#help(DecisionTreeClassifier)

In [ ]:
arbol_podado = DecisionTreeClassifier(max_depth = 2)
arbol_podado.fit(X_entreno, y_entreno)

In [ ]:
reporte_modelo(arbol_podado)

## Número Máximo de Nodos Hoja (Max Leaf Nodes)

In [ ]:
arbol_max_hojas = DecisionTreeClassifier(max_leaf_nodes = 3)
arbol_max_hojas.fit(X_entreno, y_entreno)

In [ ]:
reporte_modelo(arbol_max_hojas)

## Criterio (Criterion)

In [ ]:
arbol_entropia = DecisionTreeClassifier(criterion = 'entropy')
arbol_entropia.fit(X_entreno, y_entreno)

In [ ]:
reporte_modelo(arbol_podado)

---